# RBF-SVM with seed-matched MRI embeddings
This analysis removes the remaining seed-matching caveat from the classical SVM analysis.

Previously, every SVM seed used the same 64-dimensional MRI representation extracted from the **seed-17 neural MRI encoder**. Here, each downstream SVM run uses the MRI representation from the **corresponding full-architecture neural seed**:

- seed 17 SVM ← seed 17 MRI encoder
- seed 42 SVM ← seed 42 MRI encoder
- seed 73 SVM ← seed 73 MRI encoder
- seed 101 SVM ← seed 101 MRI encoder
- seed 137 SVM ← seed 137 MRI encoder

The outer folds remain exactly the same.

## Full-architecture embeddings

The original classical baseline uses **64-dimensional MRI embeddings**. Of the capacity variants tested this week, the original/full neural architecture has the same shared modality dimension of 64. The medium and small variants use reduced shared dimensions, so they cannot be the source of the original 64-D baseline representation.

The notebook therefore extracts the MRI branch from the **best-validation checkpoint of the full/original-capacity model** for each fold and seed.

## Provenance check

Before generating the other seeds, the notebook reconstructs the seed-17 embeddings and compares them against the existing seed-17 baseline embedding files.

If those do not match closely, the notebook stops rather than creating a different experiment.

## SVM protocol

After the 25 embedding files are generated, the notebook reruns only the RBF-SVM.

The SVM hyperparameters are selected **once per outer fold using seed 17**, then held fixed across the five matched embedding seeds. This preserves the original baseline design and avoids giving every neural seed a separate hyperparameter-selection lottery.

The final reported seed variability therefore reflects:

1. variation in the learned MRI representation across neural seeds; and
2. the small stochastic component of SVM probability estimation.


In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from IPython.display import display

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Experiment

FOLDS = [0, 1, 2, 3, 4]
SEEDS = [17, 42, 73, 101, 137]

SEARCH_SEED = 17
THRESHOLD = 0.50

MRI_DIM = 64
MRI_SPATIAL_SHAPE = (177, 213, 183)
MRI_PATH_COL = "MRI__NORMALIZED_T1_NPY_PATH"
MRI_MASK_COL = "BRANCH_MASK__MRI"

# Float32 inference is deliberate: it gives the best chance of reproducing
# the already-existing seed-17 embeddings exactly.
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
MRI_BATCH_SIZE = 1
NUM_WORKERS = 0

# Stop if regenerated seed-17 embeddings differ materially from the
# existing baseline embeddings.
STRICT_SEED17_PROVENANCE_CHECK = True
SEED17_ATOL = 1e-5

# JHub paths

DATA_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")
RUNS_ROOT = Path("/home/jovyan/thesis_runs")

INPUT_ROOT = (
    DATA_ROOT
    / "models"
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

# Five-seed original/full-capacity neural experiment.
FULL_NEURAL_ROOT = (
    RUNS_ROOT
    / "gated_cmt_fixed_equal_fusion_full_arch_md050"
)

# Existing seed-17 MRI embeddings used by the original classical baseline.
OLD_SEED17_EMBEDDING_ROOT = (
    RUNS_ROOT
    / "svm_rbf_predictions"
    / "mri_embeddings"
)

# Existing SVM search results selected on the old seed-17 embedding set.
# These are reused only if the seed-17 provenance check passes.
OLD_SVM_SEARCH_ROOT = (
    RUNS_ROOT
    / "svm_rbf_predictions_5seeds"
    / "search_results"
)

OUTPUT_ROOT = (
    RUNS_ROOT
    / "svm_rbf_seed_matched_mri_5seeds_v1"
)

EMBEDDING_ROOT = OUTPUT_ROOT / "mri_embeddings"
SEARCH_RESULTS_DIR = OUTPUT_ROOT / "search_results"
PREDICTIONS_DIR = OUTPUT_ROOT / "fold_predictions"
TABLES_DIR = OUTPUT_ROOT / "tables"

for d in [
    OUTPUT_ROOT,
    EMBEDDING_ROOT,
    SEARCH_RESULTS_DIR,
    PREDICTIONS_DIR,
    TABLES_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Prepared fold root:", INPUT_ROOT)
print("Full neural checkpoint root:", FULL_NEURAL_ROOT)
print("Old seed-17 embeddings:", OLD_SEED17_EMBEDDING_ROOT)
print("New output root:", OUTPUT_ROOT)

# Resolve legacy Colab MRI paths stored inside the prepared CSVs.
# The CSVs still contain /content/drive/MyDrive/adni_mri/... paths,
# while the same files now live under DATA_ROOT on JHub.

LEGACY_COLAB_ADNI_ROOT = "/content/drive/MyDrive/adni_mri"

def resolve_mri_path(raw_path) -> Path:
    s = str(raw_path).strip()

    # Some exported CSV strings contain an escaped extension: "\\.npy".
    s = s.replace("\\\\.npy", ".npy")

    # Remap the old Google Drive prefix to the JHub ADNI root.
    if s.startswith(LEGACY_COLAB_ADNI_ROOT):
        relative = s[len(LEGACY_COLAB_ADNI_ROOT):].lstrip("/")
        return DATA_ROOT / relative

    return Path(s)


### JHub path correction

The prepared fold CSVs still store MRI paths from the old Colab/Google Drive environment, for example:

```text
/content/drive/MyDrive/adni_mri/processed/.../file.npy
```

The analysis automatically remaps that prefix to:

```text
/home/jovyan/thesis_data/adni_mri/adni_mri/processed/.../file.npy
```

It also removes the stray escaped extension `\.npy` found in some CSV path strings.


## 1. Input check

No Google Drive is required if this cell passes.

The required assets are:

- five prepared outer-fold tables;
- 25 full-architecture best-validation checkpoints;
- the prepared `.npy` MRI volumes referenced by the fold tables;
- the old seed-17 embedding files for provenance checking.


In [ ]:
def fold_path(fold: int) -> Path:
    candidates = [
        INPUT_ROOT / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv",
        INPUT_ROOT / f"mci_prognosis_outer_fold_{fold}_categorical_encoded.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"No prepared fold CSV found for fold {fold} under {INPUT_ROOT}"
    )

def checkpoint_path(fold: int, seed: int) -> Path:
    return (
        FULL_NEURAL_ROOT
        / f"seed_{seed}"
        / "mci_prognosis"
        / f"fold_{fold}"
        / "checkpoints"
        / "best_validation_auc_checkpoint.pt"
    )

def old_seed17_embedding_path(fold: int) -> Path:
    return (
        OLD_SEED17_EMBEDDING_ROOT
        / f"mci_prognosis_fold_{fold}_seed_17_mri_embeddings.csv"
    )

missing = []

for fold in FOLDS:
    try:
        fp = fold_path(fold)
    except FileNotFoundError as exc:
        missing.append(str(exc))
        continue

    df = pd.read_csv(fp)

    for col in ["RID", MRI_MASK_COL, MRI_PATH_COL]:
        if col not in df.columns:
            missing.append(f"{fp}: missing column {col}")

    # Check at least one available MRI path per fold before the expensive work.
    if MRI_MASK_COL in df.columns and MRI_PATH_COL in df.columns:
        available = df.loc[
            pd.to_numeric(df[MRI_MASK_COL], errors="coerce").fillna(0).astype(int) == 1
        ]

        if available.empty:
            missing.append(f"fold {fold}: no MRI-available participants")
        else:
            sample_path = resolve_mri_path(available.iloc[0][MRI_PATH_COL])
            if not sample_path.exists():
                missing.append(
                    f"fold {fold}: sample MRI path does not exist: {sample_path}"
                )

    for seed in SEEDS:
        cp = checkpoint_path(fold, seed)
        if not cp.exists():
            missing.append(str(cp))

    old = old_seed17_embedding_path(fold)
    if not old.exists():
        missing.append(str(old))

if missing:
    print("PRE-FLIGHT FAILED")
    for item in missing:
        print(" -", item)

    raise FileNotFoundError(
        f"{len(missing)} required JHub asset(s) are missing."
    )

print("Input check passed.")
print("All five fold tables are present.")
print("All 25 full-architecture best checkpoints are present.")
print("The old five seed-17 embedding files are present.")
print("A sample prepared MRI path exists in every fold.")


## 2. Exact original/full-capacity MRI encoder

This is the MRI branch used by the original/full 3MT–TMC architecture:

- CNN channels: 16 → 32 → 64 → 128 → 256
- transformer dimension: 256
- 8 attention heads
- feed-forward dimension: 512
- one transformer layer
- final projection: 256 → **64**


In [ ]:
class ResidualDownsampleBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()

    def forward(self, inputs):
        pooled_inputs = self.pool(inputs)

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )

class MRIEncoder3D(nn.Module):
    def __init__(
        self,
        output_dim=64,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "patch_embedding_dim must be divisible by transformer_heads"
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = patch_embedding_dim

        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(16, 32),
            ResidualDownsampleBlock3D(32, 64),
            ResidualDownsampleBlock3D(64, 128),
            ResidualDownsampleBlock3D(128, 256),
        )

        stem_shape = tuple(
            math.ceil(d / 2)
            for d in self.input_shape
        )

        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                d // 2
                for d in patch_grid_shape
            )

        self.patch_grid_shape = patch_grid_shape
        self.number_of_patches = math.prod(
            patch_grid_shape
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(
                output_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
        )

    def forward(self, image):
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth * height * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                f"Unexpected MRI patch count: "
                f"{actual_patch_count} != {self.number_of_patches}"
            )

        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )

test_encoder = MRIEncoder3D(
    output_dim=MRI_DIM
)

n_params = sum(
    p.numel()
    for p in test_encoder.parameters()
)

print("MRI encoder parameters:", f"{n_params:,}")
print("Patch grid:", test_encoder.patch_grid_shape)
print("Number of transformer patches:", test_encoder.number_of_patches)

del test_encoder


## 3. Load only the MRI branch from a complete neural checkpoint

The checkpoint stores the complete multimodal model. We extract only state-dict entries belonging to:

```text
modality_encoders.encoders.mri.*
```

and load them strictly into the standalone MRI encoder.


In [ ]:
MRI_STATE_PREFIX_CANDIDATES = [
    "modality_encoders.encoders.mri.",
    "module.modality_encoders.encoders.mri.",
]

def torch_load_checkpoint(path: Path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )

def build_mri_encoder_from_checkpoint(
    fold: int,
    seed: int,
):
    path = checkpoint_path(
        fold,
        seed,
    )

    checkpoint = torch_load_checkpoint(
        path
    )

    if "model_state_dict" not in checkpoint:
        raise KeyError(
            f"{path}: checkpoint has no model_state_dict"
        )

    full_state = checkpoint[
        "model_state_dict"
    ]

    chosen_prefix = None

    for prefix in MRI_STATE_PREFIX_CANDIDATES:
        if any(
            key.startswith(prefix)
            for key in full_state
        ):
            chosen_prefix = prefix
            break

    if chosen_prefix is None:
        possible = [
            k for k in full_state
            if ".mri." in k
        ][:30]

        raise KeyError(
            "Could not find the MRI encoder prefix in the checkpoint. "
            f"Example MRI-like keys: {possible}"
        )

    mri_state = {
        key[len(chosen_prefix):]:
            value

        for key, value
        in full_state.items()

        if key.startswith(
            chosen_prefix
        )
    }

    encoder = MRIEncoder3D(
        output_dim=MRI_DIM
    )

    incompat = encoder.load_state_dict(
        mri_state,
        strict=True,
    )

    if incompat.missing_keys or incompat.unexpected_keys:
        raise RuntimeError(
            f"Unexpected state-dict mismatch. "
            f"Missing={incompat.missing_keys}; "
            f"unexpected={incompat.unexpected_keys}"
        )

    encoder = (
        encoder
        .to(DEVICE)
        .eval()
    )

    return encoder, {
        "checkpoint":
            str(path),

        "state_prefix":
            chosen_prefix,

        "checkpoint_epoch":
            checkpoint.get(
                "epoch",
                checkpoint.get(
                    "checkpoint_epoch"
                ),
            ),

        "checkpoint_validation_auc":
            checkpoint.get(
                "best_validation_auc",
                checkpoint.get(
                    "validation_auc",
                ),
            ),
    }

encoder, info = build_mri_encoder_from_checkpoint(
    fold=0,
    seed=17,
)

print(json.dumps(info, indent=2, default=str))
print("Loaded fold 0 / seed 17 MRI branch successfully.")

del encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 4. MRI-only dataset and extraction function

No new MRI preprocessing is performed. The notebook loads the already-prepared float32 NumPy volume with shape `(177, 213, 183)`.

Participants without MRI are retained and receive a 64-dimensional zero vector, matching the original classical-baseline contract.


In [ ]:
class MRIAvailableDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[
            index
        ]

        rid = int(
            row["RID"]
        )

        path = resolve_mri_path(
            row[
                MRI_PATH_COL
            ]
        )

        if not path.exists():
            raise FileNotFoundError(
                path
            )

        array = np.load(
            path,
            allow_pickle=False,
        )

        if array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                f"RID {rid}: expected MRI shape "
                f"{MRI_SPATIAL_SHAPE}, got {array.shape}"
            )

        array = np.asarray(
            array,
            dtype=np.float32,
        )

        tensor = torch.from_numpy(
            array[None, ...]
        )

        return {
            "RID":
                rid,

            "image":
                tensor,
        }

def load_fold_dataframe(
    fold: int,
) -> pd.DataFrame:
    df = pd.read_csv(
        fold_path(fold)
    )

    required = {
        "RID",
        MRI_MASK_COL,
        MRI_PATH_COL,
    }

    missing = sorted(
        required - set(df.columns)
    )

    if missing:
        raise KeyError(
            f"fold {fold}: missing required columns {missing}"
        )

    if len(df) != 544:
        raise ValueError(
            f"fold {fold}: expected 544 rows, found {len(df)}"
        )

    df["RID"] = pd.to_numeric(
        df["RID"],
        errors="raise",
    ).astype(int)

    df[MRI_MASK_COL] = pd.to_numeric(
        df[MRI_MASK_COL],
        errors="raise",
    ).astype(int)

    if df["RID"].duplicated().any():
        raise ValueError(
            f"fold {fold}: duplicate RIDs"
        )

    bad_masks = (
        set(
            df[MRI_MASK_COL].unique()
        )
        - {0, 1}
    )

    if bad_masks:
        raise ValueError(
            f"fold {fold}: invalid MRI masks {bad_masks}"
        )

    return df

MRI_EMBEDDING_COLS = [
    f"MRI_EMB_{i:02d}"
    for i in range(MRI_DIM)
]

def embedding_output_path(
    fold: int,
    seed: int,
) -> Path:
    return (
        EMBEDDING_ROOT
        / f"mci_prognosis_fold_{fold}_seed_{seed}_mri_embeddings.csv"
    )

def extract_embeddings(
    fold: int,
    seed: int,
    overwrite: bool = False,
) -> Path:
    out_path = embedding_output_path(
        fold,
        seed,
    )

    if (
        out_path.exists()
        and not overwrite
    ):
        print(
            f"[resume] exists: {out_path.name}"
        )
        return out_path

    df = load_fold_dataframe(
        fold
    )

    output = pd.DataFrame({
        "RID":
            df["RID"].astype(int),

        MRI_MASK_COL:
            df[MRI_MASK_COL].astype(int),
    })

    for col in MRI_EMBEDDING_COLS:
        output[col] = np.float32(0.0)

    available = df.loc[
        df[MRI_MASK_COL] == 1,
        [
            "RID",
            MRI_PATH_COL,
        ],
    ].copy()

    print(
        f"fold={fold} seed={seed}: "
        f"{len(available)} MRI-available / {len(df)} total"
    )

    encoder, ckpt_info = (
        build_mri_encoder_from_checkpoint(
            fold,
            seed,
        )
    )

    dataset = MRIAvailableDataset(
        available
    )

    loader = DataLoader(
        dataset,
        batch_size=MRI_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
    )

    rid_to_embedding = {}

    started = time.time()

    with torch.inference_mode():
        for batch_idx, batch in enumerate(
            loader
        ):
            image = (
                batch["image"]
                .to(
                    DEVICE,
                    dtype=torch.float32,
                    non_blocking=True,
                )
            )

            embedding = (
                encoder(
                    image
                )
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            rids = (
                batch["RID"]
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            for rid, vector in zip(
                rids,
                embedding,
            ):
                rid_to_embedding[
                    int(rid)
                ] = vector

            if (
                batch_idx == 0
                or (batch_idx + 1) % 50 == 0
                or (batch_idx + 1) == len(loader)
            ):
                print(
                    f"  {batch_idx + 1}/{len(loader)} batches"
                )

    matrix = output[
        MRI_EMBEDDING_COLS
    ].to_numpy(
        dtype=np.float32
    )

    rid_to_row = {
        int(rid):
            i
        for i, rid
        in enumerate(
            output["RID"]
        )
    }

    for rid, vector in rid_to_embedding.items():
        matrix[
            rid_to_row[rid],
            :
        ] = vector

    output[
        MRI_EMBEDDING_COLS
    ] = matrix

    unavailable_matrix = output.loc[
        output[MRI_MASK_COL] == 0,
        MRI_EMBEDDING_COLS,
    ].to_numpy(
        dtype=np.float32
    )

    if not np.allclose(
        unavailable_matrix,
        0.0,
        atol=0.0,
    ):
        raise RuntimeError(
            "Unavailable MRI participants are not exactly zero."
        )

    if not np.isfinite(
        output[
            MRI_EMBEDDING_COLS
        ].to_numpy()
    ).all():
        raise RuntimeError(
            "Non-finite embedding values produced."
        )

    output.to_csv(
        out_path,
        index=False,
    )

    metadata_path = (
        EMBEDDING_ROOT
        / f"mci_prognosis_fold_{fold}_seed_{seed}_metadata.json"
    )

    with open(
        metadata_path,
        "w",
    ) as f:
        json.dump(
            {
                **ckpt_info,
                "fold":
                    fold,
                "seed":
                    seed,
                "n_total":
                    int(len(df)),
                "n_mri_available":
                    int(len(available)),
                "embedding_dim":
                    MRI_DIM,
                "device":
                    str(DEVICE),
                "elapsed_seconds":
                    float(
                        time.time()
                        - started
                    ),
            },
            f,
            indent=2,
        )

    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        "Saved:",
        out_path,
    )

    return out_path


## 5. Rebuild seed 17 first and prove provenance

This is deliberately done **before** seeds 42/73/101/137.

If the new fold-specific seed-17 embeddings reproduce the files that the original SVM used, we know we have reconstructed the correct encoder output and checkpoint source.


In [ ]:
seed17_comparison_rows = []

for fold in FOLDS:
    print("\n" + "=" * 90)
    print(f"SEED-17 PROVENANCE CHECK — FOLD {fold}")
    print("=" * 90)

    new_path = extract_embeddings(
        fold=fold,
        seed=17,
        overwrite=False,
    )

    old_path = old_seed17_embedding_path(
        fold
    )

    new = pd.read_csv(
        new_path
    ).sort_values(
        "RID"
    ).reset_index(
        drop=True
    )

    old = pd.read_csv(
        old_path
    ).sort_values(
        "RID"
    ).reset_index(
        drop=True
    )

    required = {
        "RID",
        MRI_MASK_COL,
        *MRI_EMBEDDING_COLS,
    }

    old_missing = sorted(
        required - set(old.columns)
    )

    new_missing = sorted(
        required - set(new.columns)
    )

    if old_missing or new_missing:
        raise KeyError(
            f"fold {fold}: embedding-column mismatch. "
            f"old missing={old_missing}; new missing={new_missing}"
        )

    if not np.array_equal(
        new["RID"].to_numpy(),
        old["RID"].to_numpy(),
    ):
        raise ValueError(
            f"fold {fold}: RID mismatch"
        )

    if not np.array_equal(
        new[MRI_MASK_COL].to_numpy(),
        old[MRI_MASK_COL].to_numpy(),
    ):
        raise ValueError(
            f"fold {fold}: MRI-mask mismatch"
        )

    A = new[
        MRI_EMBEDDING_COLS
    ].to_numpy(
        dtype=np.float64
    )

    B = old[
        MRI_EMBEDDING_COLS
    ].to_numpy(
        dtype=np.float64
    )

    abs_diff = np.abs(
        A - B
    )

    row = {
        "FOLD":
            fold,

        "MAX_ABS_DIFF":
            float(
                abs_diff.max()
            ),

        "MEAN_ABS_DIFF":
            float(
                abs_diff.mean()
            ),

        "ALLCLOSE_ATOL_1E_5":
            bool(
                np.allclose(
                    A,
                    B,
                    rtol=0.0,
                    atol=SEED17_ATOL,
                )
            ),
    }

    seed17_comparison_rows.append(
        row
    )

    print(row)

seed17_comparison = pd.DataFrame(
    seed17_comparison_rows
)

display(
    seed17_comparison
)

seed17_comparison.to_csv(
    TABLES_DIR
    / "seed17_embedding_provenance_check.csv",
    index=False,
)

provenance_passed = bool(
    seed17_comparison[
        "ALLCLOSE_ATOL_1E_5"
    ].all()
)

print(
    "\nSeed-17 provenance passed:",
    provenance_passed,
)

if (
    STRICT_SEED17_PROVENANCE_CHECK
    and not provenance_passed
):
    raise RuntimeError(
        "STOPPING: regenerated seed-17 MRI embeddings do not match "
        "the embeddings used by the original SVM closely enough. "
        "Do not generate the other seeds until the source checkpoint/"
        "extraction definition is reconciled."
    )


## 6. Extract MRI embeddings for seeds 42, 73, 101 and 137

This produces the complete set of **25 fold × seed embedding files**.


In [ ]:
for seed in [
    42,
    73,
    101,
    137,
]:
    for fold in FOLDS:
        print("\n" + "=" * 90)
        print(
            f"EXTRACTING MRI EMBEDDINGS — seed={seed}, fold={fold}"
        )
        print("=" * 90)

        extract_embeddings(
            fold=fold,
            seed=seed,
            overwrite=False,
        )

all_embedding_files = sorted(
    EMBEDDING_ROOT.glob(
        "mci_prognosis_fold_*_seed_*_mri_embeddings.csv"
    )
)

print(
    "\nEmbedding files found:",
    len(all_embedding_files),
)

if len(all_embedding_files) != 25:
    raise RuntimeError(
        f"Expected 25 embedding files, found {len(all_embedding_files)}"
    )

print("All 25 matched fold/seed MRI embedding files are ready.")


# Part II — rerun RBF-SVM with the matched MRI representation

The remaining cells use each neural seed's own MRI features.

The non-MRI predictor contract is unchanged from the previous SVM baseline.


In [ ]:
CONTINUOUS_COLS = [
    "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
    "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
    "ADAS__TOTSCORE__Z",
    "ADAS__TOTAL13__Z",
    "MMSE__MMSE_TOTAL_SCORE__Z",
    "MMSE__MMSE_ORIENTATION_SCORE__Z",
    "MMSE__MMSE_REGISTRATION_SCORE__Z",
    "MMSE__MMSE_ATTENTION_SCORE__Z",
    "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
    "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
    "FAQ__FAQTOTAL__Z",
    "CSF__ABETA40__Z",
    "CSF__ABETA42__Z",
    "CSF__TAU__Z",
    "CSF__PTAU__Z",
    "CSF__ABETA42_40_RATIO__Z",
    "PLASMA__pT217_F__Z",
    "PLASMA__AB42_F__Z",
    "PLASMA__AB40_F__Z",
    "PLASMA__AB42_AB40_F__Z",
    "PLASMA__pT217_AB42_F__Z",
    "PLASMA__NfL_Q__Z",
    "PLASMA__GFAP_Q__Z",
    "PLASMA__NfL_F__Z",
    "PLASMA__GFAP_F__Z",
]

CATEGORICAL_COLS = [
    "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
    "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
    "APOE__APOE4_ALLELE_COUNT__IDX",
]

BRANCH_MASK_COLS = [
    "BRANCH_MASK__DEMOGRAPHICS",
    "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    "BRANCH_MASK__CSF",
    "BRANCH_MASK__PLASMA",
    "BRANCH_MASK__APOE",
]

def predictor_to_feature_mask(
    predictor: str,
) -> str:
    base = predictor

    for suffix in (
        "__Z",
        "__IDX",
    ):
        if base.endswith(
            suffix
        ):
            base = base[
                :-len(suffix)
            ]
            break

    return (
        "FEATURE_MASK__"
        + base.replace(
            "__",
            "_",
        )
    )

EXPECTED_FEATURE_MASKS = [
    predictor_to_feature_mask(c)
    for c in (
        CONTINUOUS_COLS
        + CATEGORICAL_COLS
    )
]

assert len(CONTINUOUS_COLS) == 25
assert len(CATEGORICAL_COLS) == 3
assert len(EXPECTED_FEATURE_MASKS) == 28


## 7. Build SVM matrices using the corresponding MRI seed


In [ ]:
def matched_embedding_path(
    fold: int,
    seed: int,
) -> Path:
    return embedding_output_path(
        fold,
        seed,
    )

def load_svm_fold(
    fold: int,
) -> pd.DataFrame:
    df = pd.read_csv(
        fold_path(fold)
    )

    if "MCI_PROGNOSIS_TARGET" not in df.columns:
        if "MODEL_TARGET" not in df.columns:
            raise KeyError(
                f"fold {fold}: no target column"
            )

        df[
            "MCI_PROGNOSIS_TARGET"
        ] = pd.to_numeric(
            df["MODEL_TARGET"],
            errors="raise",
        ).astype(int)

    required = {
        "RID",
        "MCI_PROGNOSIS_TARGET",
        "DATA_ROLE",
        *CONTINUOUS_COLS,
        *CATEGORICAL_COLS,
        *EXPECTED_FEATURE_MASKS,
        *BRANCH_MASK_COLS,
        MRI_MASK_COL,
    }

    missing = sorted(
        required - set(df.columns)
    )

    if missing:
        raise KeyError(
            f"fold {fold}: missing SVM columns {missing}"
        )

    df["RID"] = pd.to_numeric(
        df["RID"],
        errors="raise",
    ).astype(int)

    df[
        "MCI_PROGNOSIS_TARGET"
    ] = pd.to_numeric(
        df[
            "MCI_PROGNOSIS_TARGET"
        ],
        errors="raise",
    ).astype(int)

    df["DATA_ROLE"] = (
        df["DATA_ROLE"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Preserve the original missing-value handling.
    for predictor in CONTINUOUS_COLS:
        mask_col = predictor_to_feature_mask(
            predictor
        )

        values = pd.to_numeric(
            df[predictor],
            errors="coerce",
        )

        if (
            (
                df[mask_col].astype(int)
                == 1
            )
            & values.isna()
        ).any():
            raise ValueError(
                f"fold {fold}: observed {predictor} contains NaN"
            )

        values = values.fillna(
            0.0
        )

        values.loc[
            df[mask_col].astype(int)
            == 0
        ] = 0.0

        df[predictor] = values

    for predictor in CATEGORICAL_COLS:
        mask_col = predictor_to_feature_mask(
            predictor
        )

        df[predictor] = pd.to_numeric(
            df[predictor],
            errors="raise",
        ).astype(int)

        df.loc[
            df[mask_col].astype(int)
            == 0,
            predictor,
        ] = 0

    return df

def make_svm_matrices(
    fold: int,
    embedding_seed: int,
):
    df = load_svm_fold(
        fold
    )

    emb = pd.read_csv(
        matched_embedding_path(
            fold,
            embedding_seed,
        )
    )

    emb["RID"] = pd.to_numeric(
        emb["RID"],
        errors="raise",
    ).astype(int)

    required_emb = {
        "RID",
        MRI_MASK_COL,
        *MRI_EMBEDDING_COLS,
    }

    missing = sorted(
        required_emb
        - set(
            emb.columns
        )
    )

    if missing:
        raise KeyError(
            f"fold={fold} seed={embedding_seed}: "
            f"embedding columns missing {missing}"
        )

    if set(
        df["RID"]
    ) != set(
        emb["RID"]
    ):
        raise ValueError(
            f"fold={fold} seed={embedding_seed}: RID mismatch"
        )

    df = df.merge(
        emb[
            [
                "RID",
                *MRI_EMBEDDING_COLS,
            ]
        ],
        on="RID",
        how="left",
        validate="one_to_one",
    )

    train = df.loc[
        df["DATA_ROLE"]
        == "train"
    ].copy()

    val = df.loc[
        df["DATA_ROLE"]
        == "validation"
    ].copy()

    test = df.loc[
        df["DATA_ROLE"]
        == "test"
    ].copy()

    try:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float64,
        )
    except TypeError:
        ohe = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float64,
        )

    ohe.fit(
        train[
            CATEGORICAL_COLS
        ]
    )

    extra_cols = (
        EXPECTED_FEATURE_MASKS
        + BRANCH_MASK_COLS
        + [MRI_MASK_COL]
    )

    def transform(
        part,
    ):
        X = np.concatenate(
            [
                part[
                    CONTINUOUS_COLS
                ].to_numpy(
                    dtype=np.float64
                ),

                ohe.transform(
                    part[
                        CATEGORICAL_COLS
                    ]
                ),

                part[
                    MRI_EMBEDDING_COLS
                ].to_numpy(
                    dtype=np.float64
                ),

                part[
                    extra_cols
                ].to_numpy(
                    dtype=np.float64
                ),
            ],
            axis=1,
        )

        y = part[
            "MCI_PROGNOSIS_TARGET"
        ].to_numpy(
            dtype=int
        )

        rid = part[
            "RID"
        ].to_numpy(
            dtype=int
        )

        return X, y, rid

    X_train, y_train, rid_train = transform(
        train
    )

    X_val, y_val, rid_val = transform(
        val
    )

    X_test, y_test, rid_test = transform(
        test
    )

    return {
        "X_train":
            X_train,
        "y_train":
            y_train,
        "rid_train":
            rid_train,

        "X_val":
            X_val,
        "y_val":
            y_val,
        "rid_val":
            rid_val,

        "X_test":
            X_test,
        "y_test":
            y_test,
        "rid_test":
            rid_test,
    }

example = make_svm_matrices(
    fold=0,
    embedding_seed=17,
)

print(
    "Fold 0 / seed 17 matrix shapes:",
    example["X_train"].shape,
    example["X_val"].shape,
    example["X_test"].shape,
)

print(
    "Expected transformed feature count: 133"
)


## 8. Get the fold-level SVM hyperparameters

Preferred route: reuse the SVM configuration already selected on the original seed-17 embeddings, **but only because the provenance check passed**.

If those JSON files are unavailable, this notebook reruns the exact original SVM search on the regenerated seed-17 embeddings.


In [ ]:
SVM_C_VALUES = [
    0.1,
    1.0,
    10.0,
    100.0,
]

SVM_GAMMA_VALUES = [
    "scale",
    0.001,
    0.01,
    0.1,
]

INNER_CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEARCH_SEED,
)

SCORING = {
    "roc_auc":
        "roc_auc",

    "ap":
        "average_precision",
}

def search_selected_path(
    fold: int,
) -> Path:
    return (
        SEARCH_RESULTS_DIR
        / f"fold_{fold}_svm_selected.json"
    )

def old_selected_path(
    fold: int,
) -> Path:
    return (
        OLD_SVM_SEARCH_ROOT
        / f"fold_{fold}_svm_selected.json"
    )

def select_svm_params_for_fold(
    fold: int,
):
    new_selected = search_selected_path(
        fold
    )

    if new_selected.exists():
        with open(
            new_selected,
            "r",
        ) as f:
            payload = json.load(
                f
            )

        print(
            f"[resume] fold {fold}: "
            "using already-saved matched-run parameters"
        )

        return payload[
            "selected_params"
        ]

    old_selected = old_selected_path(
        fold
    )

    if (
        provenance_passed
        and old_selected.exists()
    ):
        with open(
            old_selected,
            "r",
        ) as f:
            old_payload = json.load(
                f
            )

        params = dict(
            old_payload[
                "selected_params"
            ]
        )

        payload = {
            "fold":
                fold,

            "source":
                "reused_from_original_seed17_svm_search_after_embedding_provenance_pass",

            "selected_params":
                params,
        }

        with open(
            new_selected,
            "w",
        ) as f:
            json.dump(
                payload,
                f,
                indent=2,
            )

        print(
            f"fold {fold}: reused original seed-17 SVM parameters:",
            params,
        )

        return params

    print(
        f"fold {fold}: original selected-parameter file unavailable; "
        "rerunning the exact seed-17 SVM search."
    )

    M = make_svm_matrices(
        fold=fold,
        embedding_seed=17,
    )

    base = SVC(
        kernel="rbf",
        probability=False,
        cache_size=2000,
    )

    grid = {
        "C":
            SVM_C_VALUES,

        "gamma":
            SVM_GAMMA_VALUES,

        "class_weight":
            [
                None,
                "balanced",
            ],
    }

    search = GridSearchCV(
        base,
        grid,
        scoring=SCORING,
        refit=False,
        cv=INNER_CV,
        n_jobs=-1,
        error_score="raise",
    )

    search.fit(
        M["X_train"],
        M["y_train"],
    )

    cv = pd.DataFrame(
        search.cv_results_
    )

    top = (
        cv.sort_values(
            [
                "mean_test_roc_auc",
                "mean_test_ap",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .head(5)
        .reset_index(
            drop=True
        )
    )

    validation_rows = []

    for rank, row in top.iterrows():
        params = dict(
            row["params"]
        )

        model = SVC(
            kernel="rbf",
            probability=False,
            cache_size=2000,
            **params,
        )

        model.fit(
            M["X_train"],
            M["y_train"],
        )

        score = model.decision_function(
            M["X_val"]
        )

        validation_rows.append({
            "rank":
                rank + 1,

            "validation_roc_auc":
                float(
                    roc_auc_score(
                        M["y_val"],
                        score,
                    )
                ),

            "validation_ap":
                float(
                    average_precision_score(
                        M["y_val"],
                        score,
                    )
                ),

            "inner_cv_roc_auc":
                float(
                    row[
                        "mean_test_roc_auc"
                    ]
                ),

            "params":
                params,
        })

    ranking = (
        pd.DataFrame(
            validation_rows
        )
        .sort_values(
            [
                "validation_roc_auc",
                "validation_ap",
                "inner_cv_roc_auc",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    params = dict(
        ranking.iloc[0][
            "params"
        ]
    )

    payload = {
        "fold":
            fold,

        "source":
            "retuned_on_regenerated_seed17_embeddings",

        "selected_params":
            params,
    }

    with open(
        new_selected,
        "w",
    ) as f:
        json.dump(
            payload,
            f,
            indent=2,
        )

    ranking_out = ranking.copy()
    ranking_out["params"] = (
        ranking_out["params"]
        .apply(
            lambda x: json.dumps(
                x,
                sort_keys=True,
            )
        )
    )

    ranking_out.to_csv(
        SEARCH_RESULTS_DIR
        / f"fold_{fold}_svm_validation_ranking.csv",
        index=False,
    )

    return params

selected_params_by_fold = {
    fold:
        select_svm_params_for_fold(
            fold
        )
    for fold in FOLDS
}

display(
    pd.DataFrame(
        [
            {
                "FOLD":
                    fold,

                **params,
            }
            for fold, params
            in selected_params_by_fold.items()
        ]
    )
)


## 9. Fit the five matched SVM pipelines

For each seed, the SVM now sees the MRI representation learned by the same neural seed.


In [ ]:
def prediction_path(
    fold: int,
    seed: int,
) -> Path:
    return (
        PREDICTIONS_DIR
        / f"svm_rbf_fold_{fold}_seed_{seed}_matched_mri_test_predictions.csv"
    )

for fold in FOLDS:
    params = selected_params_by_fold[
        fold
    ]

    for seed in SEEDS:
        path = prediction_path(
            fold,
            seed,
        )

        if path.exists():
            print(
                "[resume]",
                path.name,
            )
            continue

        M = make_svm_matrices(
            fold=fold,
            embedding_seed=seed,
        )

        model = SVC(
            kernel="rbf",
            probability=True,
            cache_size=2000,
            random_state=seed,
            **params,
        )

        model.fit(
            M["X_train"],
            M["y_train"],
        )

        prob = model.predict_proba(
            M["X_test"]
        )[:, 1]

        pred = pd.DataFrame({
            "MODEL":
                "svm_rbf_seed_matched_mri",

            "SEED":
                seed,

            "FOLD":
                fold,

            "RID":
                M["rid_test"],

            "TARGET":
                M["y_test"],

            "PROB_PMCI":
                prob,
        })

        pred[
            "PREDICTED_CLASS_0P5"
        ] = (
            pred[
                "PROB_PMCI"
            ]
            >= THRESHOLD
        ).astype(int)

        pred.to_csv(
            path,
            index=False,
        )

        print(
            "Saved:",
            path.name,
        )


## 10. Aggregate five-fold OOF results for each matched seed


In [ ]:
def metrics_from_prob(
    y,
    p,
):
    y = np.asarray(
        y,
        dtype=int,
    )

    p = np.asarray(
        p,
        dtype=float,
    )

    pred = (
        p >= THRESHOLD
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn)
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "ROC_AUC":
            float(
                roc_auc_score(
                    y,
                    p,
                )
            ),

        "AP":
            float(
                average_precision_score(
                    y,
                    p,
                )
            ),

        "BALANCED_ACCURACY":
            float(
                balanced_accuracy_score(
                    y,
                    pred,
                )
            ),

        "SENSITIVITY":
            float(sensitivity),

        "SPECIFICITY":
            float(specificity),

        "F1":
            float(
                f1_score(
                    y,
                    pred,
                )
            ),

        "BRIER":
            float(
                brier_score_loss(
                    y,
                    p,
                )
            ),
    }

pooled_frames = []
metric_rows = []

for seed in SEEDS:
    pieces = []

    for fold in FOLDS:
        pieces.append(
            pd.read_csv(
                prediction_path(
                    fold,
                    seed,
                )
            )
        )

    pooled = pd.concat(
        pieces,
        ignore_index=True,
    )

    if (
        len(pooled) != 544
        or pooled["RID"].nunique() != 544
    ):
        raise ValueError(
            f"seed {seed}: expected 544 unique OOF participants"
        )

    pooled.to_csv(
        PREDICTIONS_DIR
        / f"svm_rbf_seed_{seed}_matched_mri_pooled_oof_predictions.csv",
        index=False,
    )

    pooled_frames.append(
        pooled
    )

    metric_rows.append({
        "SEED":
            seed,

        **metrics_from_prob(
            pooled["TARGET"],
            pooled["PROB_PMCI"],
        ),
    })

seed_metrics = pd.DataFrame(
    metric_rows
)

display(
    seed_metrics
)

seed_metrics.to_csv(
    TABLES_DIR
    / "svm_matched_mri_pooled_metrics_by_seed.csv",
    index=False,
)

summary = pd.DataFrame([{
    "N_SEEDS":
        len(SEEDS),

    "ROC_AUC_MEAN":
        seed_metrics[
            "ROC_AUC"
        ].mean(),

    "ROC_AUC_SD":
        seed_metrics[
            "ROC_AUC"
        ].std(
            ddof=1
        ),

    "AP_MEAN":
        seed_metrics[
            "AP"
        ].mean(),

    "AP_SD":
        seed_metrics[
            "AP"
        ].std(
            ddof=1
        ),

    "BRIER_MEAN":
        seed_metrics[
            "BRIER"
        ].mean(),

    "BRIER_SD":
        seed_metrics[
            "BRIER"
        ].std(
            ddof=1
        ),
}])

print(
    "\nMatched end-to-end seed summary:"
)

display(
    summary
)

summary.to_csv(
    TABLES_DIR
    / "svm_matched_mri_5seed_mean_sd.csv",
    index=False,
)


## 11. Five-seed probability ensemble

This averages the five **end-to-end matched** SVM probabilities per participant, then calculates one ROC AUC across the 544 OOF participants.


In [ ]:
all_pooled = pd.concat(
    pooled_frames,
    ignore_index=True,
)

ensemble = (
    all_pooled
    .groupby("RID")
    .agg(
        TARGET=(
            "TARGET",
            "first",
        ),

        PROB_PMCI=(
            "PROB_PMCI",
            "mean",
        ),

        PROB_PMCI_SD=(
            "PROB_PMCI",
            "std",
        ),
    )
    .reset_index()
)

assert len(ensemble) == 544

ensemble_metrics = metrics_from_prob(
    ensemble["TARGET"],
    ensemble["PROB_PMCI"],
)

print(
    "Matched-MRI five-seed SVM ensemble:"
)

print(
    json.dumps(
        ensemble_metrics,
        indent=2,
    )
)

ensemble.to_csv(
    PREDICTIONS_DIR
    / "svm_matched_mri_5seed_probability_ensemble_oof.csv",
    index=False,
)

pd.DataFrame(
    [ensemble_metrics]
).to_csv(
    TABLES_DIR
    / "svm_matched_mri_5seed_probability_ensemble_metrics.csv",
    index=False,
)


## 12. Compare against the previous fixed-seed-17-MRI SVM

If the old five-seed SVM prediction files are still on JHub, this cell quantifies exactly how much of the apparent SVM stability came from freezing the MRI representation.


In [ ]:
FIXED_MRI_SVM_PREDICTIONS_DIR = (
    RUNS_ROOT
    / "svm_rbf_predictions_5seeds"
    / "fold_predictions"
)

comparison_rows = []

fixed_available = all(
    (
        FIXED_MRI_SVM_PREDICTIONS_DIR
        / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
    ).exists()
    for fold in FOLDS
    for seed in SEEDS
)

if not fixed_available:
    print(
        "Old fixed-MRI five-seed SVM predictions are not all present; "
        "skipping direct comparison."
    )
else:
    for seed in SEEDS:
        fixed_parts = []

        for fold in FOLDS:
            fixed_parts.append(
                pd.read_csv(
                    FIXED_MRI_SVM_PREDICTIONS_DIR
                    / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
                )
            )

        fixed = pd.concat(
            fixed_parts,
            ignore_index=True,
        )

        matched = pd.read_csv(
            PREDICTIONS_DIR
            / f"svm_rbf_seed_{seed}_matched_mri_pooled_oof_predictions.csv"
        )

        fixed_auc = roc_auc_score(
            fixed["TARGET"],
            fixed["PROB_PMCI"],
        )

        matched_auc = roc_auc_score(
            matched["TARGET"],
            matched["PROB_PMCI"],
        )

        comparison_rows.append({
            "SEED":
                seed,

            "FIXED_SEED17_MRI_AUC":
                fixed_auc,

            "MATCHED_MRI_AUC":
                matched_auc,

            "DELTA_MATCHED_MINUS_FIXED":
                matched_auc
                - fixed_auc,
        })

    comparison = pd.DataFrame(
        comparison_rows
    )

    display(
        comparison
    )

    comparison.to_csv(
        TABLES_DIR
        / "matched_vs_fixed_mri_svm_auc_by_seed.csv",
        index=False,
    )


## 13. Final outputs to send back

The most important files are:

```text
/home/jovyan/thesis_runs/svm_rbf_seed_matched_mri_5seeds_v1/tables/
```

Especially:

- `seed17_embedding_provenance_check.csv`
- `svm_matched_mri_pooled_metrics_by_seed.csv`
- `svm_matched_mri_5seed_mean_sd.csv`
- `svm_matched_mri_5seed_probability_ensemble_metrics.csv`
- `matched_vs_fixed_mri_svm_auc_by_seed.csv` (when the old SVM outputs are available)

For the scientific comparison, use the **matched-MRI five-seed mean ± SD** as the end-to-end SVM seed-sensitivity result.
